# Profile HMM tolpology

This notebook is just a quick implementation of our profile HMM tolpology to check if the logic is right. It includes:
+ Assigning each column in the alignment a state
+ Computing the path of states for each sequence in the alignment
+ Computing initial, transition and emission probabilities based on the alignment and states

In [6]:
from collections import defaultdict

class PHMM():

    def read_fasta(self, filename):
        sequences = []
        seq = ""

        with open(filename, "r") as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if seq:
                        sequences.append(seq)
                        seq = ""

                else:
                    seq += line

            if seq:
                sequences.append(seq)

        return sequences


    def colum_assignments(self, sequences):
        col_types = []
        num_seqs = len(sequences)
        seq_len = len(sequences[0])

        for i in range(seq_len):
            gaps = sum(1 for seq in sequences if seq[i] == '-')

            if gaps / num_seqs > 0.5:
                col_types.append("insertion")
            else:
                col_types.append("match")

        L = col_types.count("match")
    
        return col_types, L


    def sequence_states(self, sequences, col_types, L):
        seqs_states = []
        hidden_states = set()

        for sequence in sequences:
            states = ["M_0"]
            m_index = 1

            for j, char in enumerate(sequence):

                if col_types[j] == "match":
                    if char != "-":
                        states.append(f"M_{m_index}")
                    else:
                        states.append(f"D_{m_index}")
                    m_index += 1

                else:  
                    if char != "-":
                        states.append(f"I_{m_index-1}")

            states.append(f"M_{L+1}")
            seqs_states.append(states)

            for s in states:
                hidden_states.add(s)

        return seqs_states, hidden_states

    
    def initial_probabilities(self):
        return {"M_0" : 1.0}

    
    def valid_transitions(self, L):
        transitions = defaultdict(list)

        for i in range(L+1):
            if i < L:
                transitions[f"M_{i}"] = [f"M_{i+1}", f"I_{i}", f"D_{i+1}"]
                transitions[f"I_{i}"] = [f"M_{i+1}", f"I_{i}", f"D_{i+1}"]
                transitions[f"D_{i}"] = [f"M_{i+1}", f"I_{i}", f"D_{i+1}"]
            else:
                transitions[f"M_{i}"] = [f"M_{L+1}"]
                transitions[f"I_{i}"] = [f"M_{L+1}"]
                transitions[f"D_{i}"] = [f"M_{L+1}"]

        return transitions

    
    def transition_probabilities(self, seqs_states, transitions, alpha):
        transition_probs = defaultdict(lambda: defaultdict(float))

        for state_path in seqs_states:
            for i in range(len(state_path) - 1):
                current_state = state_path[i]
                next_state = state_path[i+1]
                transition_probs[current_state][next_state] += 1

        for state in transitions:
            for next_state in transitions[state]:
                transition_probs[state][next_state] += alpha

        for state in transitions:
            s = sum(transition_probs[state].values())
            for next_state in transitions[state]:
                transition_probs[state][next_state] = transition_probs[state][next_state] / s
    
        return transition_probs


    def emission_probabilities(self, col_types, seqs, L, b):

        emission_probs = defaultdict(lambda: defaultdict(int))

        # Match emissions
        idx = 1
        for i, state in enumerate(col_types):
            print(idx)
            for seq in seqs:
                if state == "match":
                    if seq[i] != '-':
                        emission_probs[f"M_{idx}"][seq[i]] += 1
            if i != len(col_types) - 1:
                if col_types[i + 1] == "match":
                    idx += 1

        # Convert to probs
        for state in emission_probs.keys():
            s = sum(emission_probs[state].values())
            for emission in emission_probs[state].keys():
                emission_probs[state][emission] = (emission_probs[state][emission] + (1 * b)) / (s + (20 * b))


        # Insertion emissions
        background_probs = defaultdict(int)
        for seq in seqs:
            for emission in seq:
                if emission != '-':
                    background_probs[emission] += 1

        # Convert to probs
        total = sum(background_probs.values())
        for emission in background_probs.keys():
            background_probs[emission] = (background_probs[emission] + (1 * b)) / (total + (20 * b))

        # Assign to emission_probs
        for i in range(L + 1):
            emission_probs[f"I_{i}"] = background_probs.copy()

        return emission_probs

# Test sequences

In [9]:
from pprint import pprint

# Directly providing a list of sequences as input instead of a file
sequences = [
    "ACD-EF",
    "A-DDEF",
    "ACDDEF"
]
amino_acids = ["A","R","N","D","C","E","Q","G","H","I","L","K","M","F","P","S","T","W","Y","V"]

phmm = PHMM()

# 1. Column assignment
col_types, L = phmm.colum_assignments(sequences)
print(f"Column Assignments: {col_types}")
print(f"Number of Match States (L): {L}")
print()

# 2. State paths
seqs_states, hidden_states = phmm.sequence_states(sequences, col_types, L)
print(f"Hidden states: {sorted(hidden_states)}")
print("State Paths:")
for i, path in enumerate(seqs_states, start=1):
    print(f"Sequence_{i}: {path}")
print()

# 3. Initial probabilties
initial_probabilities = phmm.initial_probabilities()
print("Initial Probabilities:")
pprint(initial_probabilities)
print()

# 4. Transition probabilities
transitions = phmm.valid_transitions(L)
transition_probs = phmm.transition_probabilities(seqs_states, transitions, alpha=1e-3)
print("Valid Transitions:")
pprint(transitions)
print()
print("Transition Probabilities:")
pprint(transition_probs)
print()

# 5. Emission probabilities
emission_probs = phmm.emission_probabilities(col_types, sequences, L, b=1)
print("Emission Probabilities:")
pprint(emission_probs)



Column Assignments: ['match', 'match', 'match', 'match', 'match', 'match']
Number of Match States (L): 6

Hidden states: ['D_2', 'D_4', 'M_0', 'M_1', 'M_2', 'M_3', 'M_4', 'M_5', 'M_6', 'M_7']
State Paths:
Sequence_1: ['M_0', 'M_1', 'M_2', 'M_3', 'D_4', 'M_5', 'M_6', 'M_7']
Sequence_2: ['M_0', 'M_1', 'D_2', 'M_3', 'M_4', 'M_5', 'M_6', 'M_7']
Sequence_3: ['M_0', 'M_1', 'M_2', 'M_3', 'M_4', 'M_5', 'M_6', 'M_7']

Initial Probabilities:
{'M_0': 1.0}

Valid Transitions:
defaultdict(<class 'list'>,
            {'D_0': ['M_1', 'I_0', 'D_1'],
             'D_1': ['M_2', 'I_1', 'D_2'],
             'D_2': ['M_3', 'I_2', 'D_3'],
             'D_3': ['M_4', 'I_3', 'D_4'],
             'D_4': ['M_5', 'I_4', 'D_5'],
             'D_5': ['M_6', 'I_5', 'D_6'],
             'D_6': ['M_7'],
             'I_0': ['M_1', 'I_0', 'D_1'],
             'I_1': ['M_2', 'I_1', 'D_2'],
             'I_2': ['M_3', 'I_2', 'D_3'],
             'I_3': ['M_4', 'I_3', 'D_4'],
             'I_4': ['M_5', 'I_4', 'D_5'],
 